# Caso Práctico: Análisis de Éxito Cinematográfico mediante Joins

## 🎯 Objetivos
En este laboratorio aplicaremos los conceptos de `merge` para resolver un problema de análisis real:
- Integrar datos descriptivos y métricas de rendimiento de películas.
- Implementar un flujo de limpieza de datos post-unión.
- Crear una métrica personalizada de "Éxito" combinando datos de ambas tablas.
- Identificar "Gemas Ocultas": películas con alta calificación pero pocos votos.

## 📖 El Problema de Negocio

Como analistas de una plataforma de streaming, queremos recomendar películas que sean de alta calidad (`mean_vote` alto) pero que no sean masivamente conocidas (`total_votes` bajo). 

Para lograrlo, necesitamos unir la tabla de **Títulos y Géneros** con la tabla de **Calificaciones**, filtrar el ruido y calcular la métrica de éxito.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path(".")

# Carga optimizada
df_movies = pd.read_csv(DATA_PATH / "IMDb movies.csv", low_memory=False)
df_ratings = pd.read_csv(DATA_PATH / "IMDb ratings.csv")

df_movies = df_movies[['imdb_title_id', 'title', 'year', 'genre', 'country']]
df_ratings = df_ratings[['imdb_title_id', 'total_votes', 'mean_vote']]

## Paso 1: Integración de Datos

Utilizaremos un **Left Join**. ¿Por qué? Porque queremos mantener el catálogo completo de películas, incluso aquellas que aún no han recibido calificaciones.

In [ ]:
df_master = df_movies.merge(df_ratings, on='imdb_title_id', how='left')

print(f"Tamaño del dataset integrado: {df_master.shape}")
display(df_master.head())

## Paso 2: Limpieza y Tratamiento de Nulos

Al hacer un Left Join, las películas sin calificación aparecen como `NaN`. Para nuestro análisis, convertiremos estos nulos en 0 para evitar errores en los cálculos matemáticos.

In [ ]:
# Llenar nulos en métricas
df_master['total_votes'] = df_master['total_votes'].fillna(0)
df_master['mean_vote'] = df_master['mean_vote'].fillna(0)

print("Valores nulos restantes:")
print(df_master[['total_votes', 'mean_vote']].isna().sum())

## Paso 3: Creación de la Métrica "Gema Oculta"

Definimos una "Gema Oculta" como una película que cumple:
1. Calificación promedio $\ge 8.0$.
2. Total de votos $\le 1000$ (para asegurar que sea "poco conocida").

In [ ]:
# Filtrado de gemas ocultas
gemas_ocultas = df_master[(df_master['mean_vote'] >= 8.0) & 
                          (df_master['total_votes'] <= 1000) & 
                          (df_master['total_votes'] > 10)] # Evitamos pelis con muy pocos votos

print(f"Se encontraron {len(gemas_ocultas)} gemas ocultas.")
display(gemas_ocultas[['title', 'genre', 'mean_vote', 'total_votes']].head(10))

## Paso 4: Análisis por Género

¿Qué géneros tienden a producir más "Gemas Ocultas"? Utilizaremos un `groupby` sobre el resultado del `merge`.

In [ ]:
# Contar gemas por género
# Nota: Un género puede tener varios valores (ej. 'Drama, Romance'). 
# Para simplicidad, tomaremos el primer género listado.
df_master['main_genre'] = df_master['genre'].str.split(',').str[0]

genre_gems = gemas_ocultas.groupby('main_genre').size().sort_values(ascending=False)

print("--- Top Géneros con más Gemas Ocultas ---")
display(genre_gems.head(10))

## 📝 Desafíos Adicionales

**Desafío 1**: Modifica el join para que sea un `Inner Join`. ¿Cuántas películas se pierden en el proceso? ¿Por qué ocurre esto?

**Desafío 2**: Crea una nueva columna llamada `weighted_score` que sea: $\text{mean\_vote} \times \log_{10}(\text{total\_votes} + 1)$. Ordena el dataset por este score y muestra las 5 películas más exitosas de la historia.

In [ ]:
# Solución Desafío 1
pass

In [ ]:
# Solución Desafío 2
pass

## 📋 Conclusiones del Caso

1. **El merge es la puerta de entrada**: Sin la unión correcta, no podemos correlacionar características (género) con resultados (votos).
2. **La limpieza post-join es obligatoria**: Los `NaN` resultantes de los Left/Outer joins pueden romper cualquier cálculo matemático.
3. **La potencia de la combinación**: La verdadera ciencia de datos ocurre cuando combinamos `merge` $\rightarrow$ `fillna` $\rightarrow$ `filter` $\rightarrow$ `groupby`.